# MolSanity — full-scale audit run on a free Colab GPU



Runs the complete `configs/full.yaml` sweep (150 epochs, 50 IG steps, 100 audited

molecules/cell, scaffold **and** random splits) across every reachable dataset ×

backbone × attributor — the publication-scale numbers the CPU dev box only

approximated.



### How to run

1. **Runtime → Change runtime type → T4 GPU** (free tier is enough — the models are

   ~43k params, <2 GB VRAM).

2. **Runtime → Run all.**



The pipeline is **resumable**: trained checkpoints + stage `.done` markers are

reused, so if the Colab session drops, just re-run the *Run the sweep* cell and it

continues where it left off (no retraining).



Datasets needing the pre-2.5 PyG stack (ShapeGGen/GraphXAI) stay blocked-tolerant

and are skipped + logged, never faked.

## 1. Verify the GPU

In [ ]:
import torch, subprocess
smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(smi or 'nvidia-smi not found')
if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU detected. Set Runtime > Change runtime type > T4 GPU, then Run all.'
    )
print('CUDA:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2. Clone the repo



If the repository is **private**, create a GitHub personal access token (repo

scope) and paste it into `GITHUB_TOKEN` below. Leave it blank for a public repo.

In [ ]:
import os, subprocess
GITHUB_TOKEN = ''  # <- paste a token here only if the repo is private
OWNER, REPO, BRANCH = 'Kar488', 'molsanity', 'claude/molsanity-scaffold-mutag-uayx7a'

auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
url = f'https://{auth}github.com/{OWNER}/{REPO}.git'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', url, REPO],
                   check=True)
os.chdir(REPO)
print('cwd:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-1'])

## 3. Install dependencies



Colab ships a CUDA build of PyTorch; PyG (2.5+), RDKit, and Captum are pure-Python

wheels that need no compiled extensions. PyTDC (for DILI/hERG/Tox21) is installed

with a minimal footprint so it doesn't fight Colab's pinned numpy/pandas.

In [ ]:
%pip install -q torch-geometric rdkit captum pyyaml
# Minimal PyTDC: avoid its heavy optional deps (transformers/scanpy/...).
%pip install -q --no-deps PyTDC huggingface_hub httpx fuzzywuzzy
print('deps installed')

In [ ]:
%pip install -q -e .
print('molsanity installed (editable)')

## 4. Smoke check — imports + a real dataset load

In [ ]:
import molsanity
from molsanity.data.datasets import load_dataset
ld = load_dataset('MUTAG')
print('MUTAG:', len(ld.dataset), 'graphs — pipeline ready')

## 5. Run the sweep



This trains every backbone on the GPU and computes the full audit battery. On a

free T4 expect roughly 1–2 hours. **Resumable** — re-run this cell if the session

drops.

In [ ]:
import time, subprocess
t0 = time.time()
proc = subprocess.run(
    ['python', '-m', 'molsanity.run_all', '--config', 'configs/full.yaml'],
    text=True,
)
print(f'\nfinished in {(time.time()-t0)/60:.1f} min (exit {proc.returncode})')

## 6. Results

In [ ]:
from pathlib import Path
for name in ['RESULTS.md', 'BENCHMARK.md']:
    p = Path(name)
    if p.exists():
        print('=' * 80, '\n', name, '\n', '=' * 80)
        print(p.read_text())

## 7. Download the outputs



Bundles the reports, run logs, checkpoints, and figures so you can pull the

full-scale results (and the retrained GPU checkpoints) off Colab.

In [ ]:
import shutil, os
shutil.make_archive('molsanity_full_run', 'zip', root_dir='.',
                    base_dir=None)  # zips the repo tree
# Trim to the artifacts that matter if the full zip is large:
print('zip size (MB):', round(os.path.getsize('molsanity_full_run.zip')/1e6, 1))
try:
    from google.colab import files
    files.download('molsanity_full_run.zip')
except Exception as e:
    print('Not on Colab or download blocked:', e)